# Aim
The aim of this notebook is to prototype an analysis pipeline for Thomas's data. Neurons, cilia all that we can wish for.

## Overview
3D Immunos for neurite, cilia, nuclei, (additionally basal body) - 


# 1. Helper functions

In [1]:
import limoncello as lc
import pyclesperanto_prototype as cle
from imaris_ims_file_reader.ims import ims
import numpy as np
import os
import re
from tqdm import tqdm
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
import napari
from skimage.morphology import skeletonize, medial_axis
from skan.csr import skeleton_to_csgraph, Skeleton, summarize
from skan import draw
from skimage.feature import shape_index
from skimage.measure import regionprops
from scipy.spatial import cKDTree
import pandas as pd
from stackview import insight
from scipy.ndimage import distance_transform_edt

In [ ]:
ims_path = r"H:\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims"

a, meta = lc.load_image(ims_path)



Opening .ims file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims
Opening readonly file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims 

Shape (T, C, Z, Y, X): (1, 4, 31, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (31, 2040, 2040)
Z Resolution: 0.484
XY Resolution: (0.305, 0.305)


If for some reason the meta data is not found in the .ims, in many Imaris exports there is a **metadata text file** next to the `.ims` file,
for example:
- `METH-PLTurbID-2h-pfa_2026-02-12_12.33.21.ims`
- `METH-PLTurbID-2h-pfa_2026-02-12_12.33.21_metadata.txt`

This step:
- Looks for a matching `*_metadata.txt` file next to your `.ims`
- Reads voxel sizes (pixel size in µm, Z-step in µm) and channel information
- Stores these values so later steps can report **real volumes in µm³** instead of just voxel counts.

If no metadata file is found, the notebook will keep using default voxel sizes (1.0 µm).

In [7]:
meta_text = lc.load_ims_metadata(ims_path)

No metadata .txt file found next to the .ims file.


watch out as sometimes the format is inconsistent and resolutions are not found in the txt file they will default to 1

Select GPU for clesperanto library

In [3]:
print(cle.available_device_names(dev_type="gpu"))
cle.select_device('NVIDIA GeForce RTX 4090')  # optional but good practice
print(cle.get_device())

['NVIDIA GeForce RTX 4090', 'gfx1036']
<NVIDIA GeForce RTX 4090 on Platform: NVIDIA CUDA (1 refs)>


In [2]:
# Simple default colormaps for a few channels
DEFAULT_COLORMAPS = ["yellow", "green", "red", "blue", "magenta", "cyan"]
CHANNEL_NAMES = ["cilia", "neurites", "basal_bodies", "nuclei"]
scale = (meta['voxel_size'])
viewer = napari.Viewer(ndisplay=3)
viewer.dims.axis_labels = ['Z', 'Y', 'X']
def set_layer_scale(event):
    layer = event.value
    if hasattr(layer, 'scale'):
        layer.scale = scale

# Connect the function to layer insertion events
viewer.layers.events.inserted.connect(set_layer_scale)

T, C, Z, Y, X = a.shape

for c in range(C):
    #volume = (volumes[c] - volumes[c].min()) / (volumes[c].max() - volumes[c].min())  # min - max normalization 
    volume = a[0,c]
    cmap = DEFAULT_COLORMAPS[c % len(DEFAULT_COLORMAPS)]
    viewer.add_image(volume, name=f"Channel {CHANNEL_NAMES[c]}", colormap=cmap, blending="additive", scale=scale, units='um')

print("Napari viewer opened. Use the sliders to explore. Close the window to continue.")
napari.run()


NameError: name 'meta' is not defined

In [3]:
# ── 3D .ims Annotator ───────────────────────────────────────────────────────
# pip install napari magicgui tifffile numpy

import napari
import numpy as np
import tifffile

from pathlib import Path
from magicgui.widgets import ComboBox, PushButton, Label, Container

# ── Config ───────────────────────────────────────────────────────────────────
IMAGE_FOLDER = r"H:\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1"  # <-- root folder
SAVE_FOLDER  = r"C:\Users\qfavey\Documents\Thomas-test-data\masks"                             # <-- change this
LABELS       = ["dendrite", "axon", "soma", "background"]

DEFAULT_COLORMAPS = ["yellow", "green", "red", "blue", "magenta", "cyan"]
CHANNEL_NAMES     = ["cilia", "neurites", "basal_bodies", "nuclei"]

# ── Find all .ims files recursively ──────────────────────────────────────────
image_paths = sorted(Path(IMAGE_FOLDER).rglob("*.ims"))
print(f"Found {len(image_paths)} .ims files")
assert len(image_paths) > 0, "No .ims files found — check IMAGE_FOLDER"

state = {"idx": 0}
Path(SAVE_FOLDER).mkdir(parents=True, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_existing_mask(path, spatial_shape, scale):
    mask_path = Path(SAVE_FOLDER) / (path.stem + "_mask.tif")
    if mask_path.exists():
        print(f"Resuming mask: {mask_path.name}")
        return tifffile.imread(mask_path)
    return np.zeros(spatial_shape, dtype=np.uint8)

def save_current():
    path = image_paths[state["idx"]]
    mask = viewer.layers["annotation"].data.astype(np.uint8)
    out  = Path(SAVE_FOLDER) / (path.stem + "_mask.tif")
    tifffile.imwrite(out, mask)
    status_label.value = f"✅ Saved → {out.name}"

# ── napari viewer (3D mode, real voxel scale) ─────────────────────────────────
viewer = napari.Viewer(ndisplay=3)
viewer.dims.axis_labels = ['Z', 'Y', 'X']

def load_current():
    path = image_paths[state["idx"]]
    a, meta = lc.load_image(str(path))
    scale   = meta['voxel_size']

    T, C, Z, Y, X = a.shape
    spatial_shape  = (Z, Y, X)
    mask           = load_existing_mask(path, spatial_shape, scale)

    viewer.layers.clear()

    # Add a scale setter so any newly inserted layer auto-gets the right scale
    def set_layer_scale(event):
        layer = event.value
        if hasattr(layer, 'scale'):
            layer.scale = scale
    viewer.layers.events.inserted.connect(set_layer_scale)

    # Add each channel as a separate image layer
    for c in range(C):
        volume = a[0, c]   # T=0, channel=c → (Z, Y, X)
        cmap   = DEFAULT_COLORMAPS[c % len(DEFAULT_COLORMAPS)]
        name   = CHANNEL_NAMES[c] if c < len(CHANNEL_NAMES) else f"ch_{c}"
        viewer.add_image(
            volume, name=name,
            colormap=cmap, blending="additive",
            scale=scale, units='um'
        )

    # Add annotation layer on top
    viewer.add_labels(mask, name="annotation",
                      num_colors=len(LABELS),
                      scale=scale)

    label_select.value = LABELS[0]
    viewer.layers["annotation"].selected_label = 1
    status_label.value = (
        f"📂 {path.name}  [{state['idx']+1}/{len(image_paths)}]\n"
        f"shape: {a.shape}  scale: {scale}"
    )

# ── Widgets ───────────────────────────────────────────────────────────────────
label_select = ComboBox(choices=LABELS, label="Class")
status_label  = Label(value="Loading...")

@label_select.changed.connect
def on_label_change(val):
    viewer.layers["annotation"].selected_label = LABELS.index(val) + 1

btn_prev      = PushButton(text="◀ Prev")
btn_next      = PushButton(text="Next ▶")
btn_save      = PushButton(text="💾 Save")
btn_save_next = PushButton(text="💾 Save & Next")

@btn_prev.clicked.connect
def prev_image():
    if state["idx"] > 0:
        state["idx"] -= 1; load_current()

@btn_next.clicked.connect
def next_image():
    if state["idx"] < len(image_paths) - 1:
        state["idx"] += 1; load_current()

@btn_save.clicked.connect
def on_save(): save_current()

@btn_save_next.clicked.connect
def on_save_next():
    save_current()
    if state["idx"] < len(image_paths) - 1:
        state["idx"] += 1; load_current()

# ── Keyboard shortcuts ────────────────────────────────────────────────────────
@viewer.bind_key("d")
def key_save_next(v): on_save_next()

@viewer.bind_key("s")
def key_save(v): save_current()

# ── Dock panel ────────────────────────────────────────────────────────────────
panel = Container(widgets=[
    status_label, label_select,
    btn_save, btn_save_next,
    btn_prev, btn_next
])
viewer.window.add_dock_widget(panel, name="Annotator", area="right")

load_current()
napari.run()

Found 22 .ims files


Opening .ims file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims
Opening readonly file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims 

Shape (T, C, Z, Y, X): (1, 4, 31, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (31, 2040, 2040)
Z Resolution: 0.484
XY Resolution: (0.305, 0.305)


TypeError: add_labels() got an unexpected keyword argument 'num_colors'

Opening .ims file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F01.ims
Opening readonly file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F01.ims 

Shape (T, C, Z, Y, X): (1, 4, 31, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (31, 2040, 2040)
Z Resolution: 0.484
XY Resolution: (0.305, 0.305)
Closing file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F01.ims 



In [5]:
# ── 3D Image Annotator (.ims) ───────────────────────────────────────────────
# pip install napari magicgui tifffile numpy

import napari
import numpy as np
import tifffile

from pathlib import Path
from magicgui.widgets import ComboBox, PushButton, Label, Container

# ── Config ──────────────────────────────────────────────────────────────────
IMAGE_FOLDER = r"H:\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1"  # <-- root folder
SAVE_FOLDER  = r"C:\Users\qfavey\Documents\Thomas-test-data\masks"                             # <-- change this
LABELS       = ["dendrite", "axon", "soma", "background"]

# Channel display settings — edit to match your staining
# (name, colormap) in channel order from lc.load_image
CHANNELS = [
    ("DAPI",  "blue"),
    ("CG",    "green"),
    ("PCNT",  "red"),
    ("TUJ1",  "gray"),
]

# ── Find all .ims files recursively ─────────────────────────────────────────
image_paths = sorted(Path(IMAGE_FOLDER).rglob("*.ims"))
print(f"Found {len(image_paths)} .ims files")
assert len(image_paths) > 0, "No .ims files found — check IMAGE_FOLDER"

state = {"idx": 0, "progress": {}}
Path(SAVE_FOLDER).mkdir(parents=True, exist_ok=True)

# ── Loaders ──────────────────────────────────────────────────────────────────
def load_image(path):
    a, meta = lc.load_image(str(path))
    # a is expected to be (C, Z, Y, X) or (Z, Y, X)
    state["meta"] = meta
    return a

def load_existing_mask(path, spatial_shape):
    mask_path = Path(SAVE_FOLDER) / (path.stem + "_mask.tif")
    if mask_path.exists():
        print(f"Resuming mask: {mask_path.name}")
        return tifffile.imread(mask_path)
    return np.zeros(spatial_shape, dtype=np.uint8)

# ── napari viewer ────────────────────────────────────────────────────────────
viewer = napari.Viewer(title="3D .ims Annotator")

def load_current():
    path = image_paths[state["idx"]]
    a    = load_image(path)

    # Handle both (C,Z,Y,X) and (Z,Y,X)
    if a.ndim == 4:
        spatial_shape = a.shape[1:]   # (Z, Y, X)
        multichannel  = True
    else:
        spatial_shape = a.shape       # (Z, Y, X)
        multichannel  = False

    mask = load_existing_mask(path, spatial_shape)

    viewer.layers.clear()

    if multichannel:
        for i, (ch_name, cmap) in enumerate(CHANNELS[:a.shape[0]]):
            viewer.add_image(
                a[i], name=ch_name,
                colormap=cmap, blending="additive",
                visible=True
            )
    else:
        viewer.add_image(a, name=path.name,
                         colormap="green", blending="additive")

    viewer.add_labels(mask, name="annotation",
                      num_colors=len(LABELS))

    label_select.value = LABELS[0]
    viewer.layers["annotation"].selected_label = 1
    status_label.value = (
        f"📂 {path.name}\n"
        f"[{state['idx']+1}/{len(image_paths)}]  "
        f"shape: {a.shape}"
    )

def save_current():
    path = image_paths[state["idx"]]
    mask = viewer.layers["annotation"].data.astype(np.uint8)
    out  = Path(SAVE_FOLDER) / (path.stem + "_mask.tif")
    tifffile.imwrite(out, mask)
    state["progress"][path.name] = "done"
    status_label.value = f"✅ Saved → {out.name}"

# ── Widgets ──────────────────────────────────────────────────────────────────
label_select = ComboBox(choices=LABELS, label="Class")
status_label  = Label(value="Loading...")

@label_select.changed.connect
def on_label_change(val):
    viewer.layers["annotation"].selected_label = LABELS.index(val) + 1

btn_prev      = PushButton(text="◀ Prev")
btn_next      = PushButton(text="Next ▶")
btn_save      = PushButton(text="💾 Save")
btn_save_next = PushButton(text="💾 Save & Next")

@btn_prev.clicked.connect
def prev_image():
    if state["idx"] > 0:
        state["idx"] -= 1; load_current()

@btn_next.clicked.connect
def next_image():
    if state["idx"] < len(image_paths) - 1:
        state["idx"] += 1; load_current()

@btn_save.clicked.connect
def on_save(): save_current()

@btn_save_next.clicked.connect
def on_save_next():
    save_current()
    if state["idx"] < len(image_paths) - 1:
        state["idx"] += 1; load_current()

# ── Keyboard shortcuts ───────────────────────────────────────────────────────
@viewer.bind_key("d")
def key_save_next(v): on_save_next()

@viewer.bind_key("s")
def key_save(v): save_current()

# ── Dock widget ──────────────────────────────────────────────────────────────
panel = Container(widgets=[
    status_label, label_select,
    btn_save, btn_save_next,
    btn_prev, btn_next
])
viewer.window.add_dock_widget(panel, name="Annotator", area="right")

load_current()
napari.run()

Found 22 .ims files
Opening .ims file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims
Opening readonly file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims 

Shape (T, C, Z, Y, X): (1, 4, 31, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (31, 2040, 2040)
Z Resolution: 0.484
XY Resolution: (0.305, 0.305)


AxisError: axis 2 is out of bounds for array of dimension 2